In [ ]:
import os

import cmocean
import h5py
import numpy as np
from matplotlib import pyplot as plt

# Visualize ESDC gap-filling evaluation samples from the HDF5 dataset.
#
# Layout: one row per sample, 7 columns:
#   t0 | t1 | t2 (gappy) | t3 | t4 | GT clean | gap mask
#
# The gap-filling dataset stores:
#   X[i, :, :, :, 0]  -- LST input sequence; middle frame (t=2) has NaN gaps
#   y[i, 0, :, :, 0]  -- original LST values at gap locations (non-gap pixels are NaN)
# Ground truth is reconstructed by combining valid X middle pixels with y gap pixels.

_gap_filling_dir = os.environ.get(
    "ESDC_GAP_FILLING_DIR",
    "../../../esdc/data/lst_gap_filling",
)
h5_path = os.path.join(_gap_filling_dir, "dataset.h5")

n_samples = 5
split = "train"   # "train" or "test"
seed = None          # set to None for non-deterministic sampling

# Must match the normalization used during dataset creation
LST_MIN, LST_MAX = 180.0, 360.0

N_TIMESTEPS = 5
MIDDLE_T = N_TIMESTEPS // 2  # index 2

CMAP = cmocean.cm.matter_r.copy()
VMIN, VMAX = 273.15, 323.15   # display range in Kelvin


def denorm_lst(arr):
    """Convert normalized [0, 1] LST back to Kelvin."""
    return arr * (LST_MAX - LST_MIN) + LST_MIN


def decode_time(t):
    return t.decode() if isinstance(t, bytes) else str(t)


# --- load and sample ---

with h5py.File(h5_path, "r") as f:
    N = f[f"X_{split}"].shape[0]
    rng = np.random.default_rng(seed)
    indices = np.sort(rng.choice(N, size=min(n_samples, N), replace=False))

    X      = f[f"X_{split}"][indices]     # (n, T, H, W, C)
    y      = f[f"y_{split}"][indices]     # (n, 1, H, W, 1)
    times  = f[f"time_{split}"][indices]  # (n, 2) byte strings
    lats   = f[f"lat_{split}"][indices]   # (n, H)
    lons   = f[f"lon_{split}"][indices]   # (n, W)

n_found = len(X)


# --- plot ---

N_COLS = N_TIMESTEPS + 2  # t0..t4  +  GT  +  gap mask
fig, axes = plt.subplots(n_found, N_COLS, figsize=(3.5 * N_COLS, 4.2 * n_found))
if n_found == 1:
    axes = [axes]

for row in range(n_found):
    lst_norm = X[row, :, :, :, 0]        # (T, H, W)
    lst_K    = denorm_lst(lst_norm)       # (T, H, W), Kelvin

    target_norm = y[row, 0, :, :, 0]     # (H, W), gap-area GT values
    target_K    = denorm_lst(target_norm) # (H, W)

    gap_mask = np.isnan(lst_K[MIDDLE_T]) # True where middle frame has synthetic gaps

    # Reconstruct the full clean middle frame:
    # valid non-gap pixels come from X; gap pixels come from y.
    gt_K = np.where(gap_mask, target_K, lst_K[MIDDLE_T])

    t_start = decode_time(times[row][0])[:10]
    t_end   = decode_time(times[row][1])[:10]
    lat_range = f"{lats[row].min():.1f}\u00b0\u2013{lats[row].max():.1f}\u00b0N"
    lon_range = f"{lons[row].min():.1f}\u00b0\u2013{lons[row].max():.1f}\u00b0E"

    # -- time series columns (t0 .. t4) --
    for t in range(N_TIMESTEPS):
        ax = axes[row][t]
        frame = np.ma.masked_invalid(lst_K[t])
        frac  = 100.0 * float(np.isnan(lst_K[t]).sum()) / lst_K[t].size

        ax.imshow(frame, cmap=CMAP, vmin=VMIN, vmax=VMAX)
        label = f"t={t}" + (" (gappy)" if t == MIDDLE_T else "")
        ax.set_title(f"{label}\ngaps={frac:.1f}%", fontsize=7)
        ax.axis("off")

    # -- ground truth column --
    ax_gt = axes[row][N_TIMESTEPS]
    gt_masked = np.ma.masked_invalid(gt_K)
    frac_missing = 100.0 * float(np.isnan(gt_K).sum()) / gt_K.size
    ax_gt.imshow(gt_masked, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax_gt.set_title(
        f"Ground truth (t=2)\n(original clean)\nmissing={frac_missing:.1f}%",
        fontsize=7,
    )
    ax_gt.axis("off")

    # -- gap mask column --
    ax_mask = axes[row][N_TIMESTEPS + 1]
    coverage = 100.0 * float(gap_mask.sum()) / gap_mask.size
    ax_mask.imshow(gap_mask.astype(float), cmap="Reds", vmin=0, vmax=1)
    ax_mask.set_title(f"Gap mask\ncoverage={coverage:.1f}%\n(red=gap)", fontsize=7)
    ax_mask.axis("off")

    # row label: sample index, location, time window
    axes[row][0].set_ylabel(
        f"sample {indices[row]}\n{lat_range}\n{lon_range}\n{t_start}\u2013{t_end}",
        fontsize=7, labelpad=4,
    )
    axes[row][0].yaxis.set_visible(True)

fig.suptitle(
    f"ESDC gap-filling evaluation dataset  ({n_found} samples, split={split!r})\n"
    "Middle frame (t=2) has synthetic NaN gaps (cyan). "
    "Green-bordered column is the reconstructed ground truth (Kelvin).",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()
plt.show()
